In [9]:
!pip install mord

In [10]:
import pandas as pd
import numpy as np
import joblib
import os
from mord import LogisticIT
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import classification_report, accuracy_score

print("Loading dataset for Ordinal Logistic Regression with pooled cross-validation...")
file_path = 'TRAINING DATA-SOIL.xlsx'
df = pd.read_excel(file_path, sheet_name=0)

# Identify temperature column name variation safely
temp_col = 'Avg_3months_Temperature_before_data_collection' if 'Avg_3months_Temperature_before_data_collection' in df.columns else 'Avg_3moths_Temperature_before_data_collection'

# Define features and target
feature_cols = [
    'LATITUDE', 'Longitude', 'Elevation_m', 'Soil_pH', 'SOC_percent', 'Clay_percent',
    'Avg_3moths_rainfall_before_data_collection_mm', temp_col,
    'Distance_from_closest_mine_m', 'Distance_upstream_mine_m',
    'Distance_from_closest_stream_m', 'Distance_from_closest_road_m',
    'Distance from closest waste disposal m', 'Distance_upstream_waste_pit_m',
    'Land_use', 'Soil_type'
]

target_col = 'Contamination_Level' if 'Contamination_Level' in df.columns else df.columns[-1]

X = df[feature_cols].copy()
y = df[target_col].astype(str)

# Handle missing values if any
X = X.fillna(X.median(numeric_only=True))

# Encode categorical variables using LabelEncoder for features
from sklearn.preprocessing import LabelEncoder
for col in ['Land_use', 'Soil_type']:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))

# 1. Apply strict ordinal mapping (DO NOT use LabelEncoder for target)
order = {
    "Clean": 0, 
    "Slight contamination": 1,
    "Moderate contamination": 2, 
    "Heavy contamination": 3
}
y_ord = y.map(order)

# Verify mapping completeness
if y_ord.isnull().any():
    print("Warning: Some target category names did not match the ordinal dictionary keys. Unique target values found:", y.unique())

# 2. Build pipeline with RobustScaler and Ordinal Logistic Regression
pipe = make_pipeline(RobustScaler(), LogisticIT(alpha=1.0))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. Generate pooled predictions across all folds
pred = cross_val_predict(pipe, X, y_ord, cv=cv)

# 4. Evaluate pooled metrics
print("\n--- Pooled 5-Fold Cross-Validation Results ---")
print("Accuracy:", round(accuracy_score(y_ord, pred), 4))
print("\nClassification Report:")
print(classification_report(y_ord, pred, target_names=["Clean", "Slight", "Moderate", "Heavy"]))

# Fit final model on all data for saving artifact if needed
pipe.fit(X, y_ord)
os.makedirs('models', exist_ok=True)
joblib.dump(pipe, 'models/soil_contamination_ordinal_logistic.pkl')
print("\nOrdinal model pipeline successfully saved to 'models/soil_contamination_ordinal_logistic.pkl'")

Loading dataset for Ordinal Logistic Regression with pooled cross-validation...

--- Pooled 5-Fold Cross-Validation Results ---
Accuracy: 0.7951

Classification Report:
              precision    recall  f1-score   support

       Clean       1.00      0.20      0.33         5
      Slight       0.62      0.50      0.56        10
    Moderate       0.00      0.00      0.00        15
       Heavy       0.81      0.99      0.89        92

    accuracy                           0.80       122
   macro avg       0.61      0.42      0.44       122
weighted avg       0.70      0.80      0.73       122



/home/user/anaconda3/envs/kakamega_hazard/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/user/anaconda3/envs/kakamega_hazard/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/user/anaconda3/envs/kakamega_hazard/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, 


Ordinal model pipeline successfully saved to 'models/soil_contamination_ordinal_logistic.pkl'
